# UNICEF Malawi – Data Pipeline

This notebook implements the full preprocessing pipeline for the UNICEF Malawi MICS dataset.
The pipeline is structured in clearly separated, independently testable sections:

1. **Imports & column group definitions** – single source of truth for all feature lists
2. **Target engineering** – binarising `FCF26` (depression)
3. **Skip-pattern fixes** – resolving survey-instrument-specific missing values before imputation
4. **Feature engineering** – creating composite / aggregated features
5. **Group imputation** – geography-stratified median / mode imputation (train-only fit)
6. **Binary encoding** – YES / NO → 0 / 1
7. **Sklearn preprocessing pipeline** – scaling, ordinal encoding, one-hot encoding
8. **`build_Xy` orchestrator** – end-to-end entry point
9. **Model training demo** – logistic regression & random forest baselines


## Section 0 – Imports & Column Group Definitions

All feature lists are declared here as **module-level constants** so that any change
propagates automatically through every downstream step.
To exclude a feature from modelling, comment it out of the relevant list.


In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import classification_report

In [2]:
# ---------------------------------------------------------------------------
# NUMERIC_COLS – continuous / interval-scale variables fed to StandardScaler.
# CB3  : child age (years)
# WB4  : mother's age
# MA2  : husband's age (0 if not married)
# LS2  : life-satisfaction numeric score
# wscore: household wealth score
# CSURV / CDEAD : surviving / deceased children of mother
# CL3  / CL13  : hours of paid labour / household work per week
# WS4  : minutes to water source (0 if on-premises)
# CB5B_num : school grade as integer (engineered from CB5B; see fix_skip_patterns)
# ---------------------------------------------------------------------------
NUMERIC_COLS = [
    'CB3', 'WB4', 'MA2', 'LS2', 'wscore', 'CSURV', 'CDEAD',
    'CL3', 'CL13', 'WS4', 'CB5B_num',
]

# ---------------------------------------------------------------------------
# ORDINAL_COLS_SPEC – ordered-categorical variables together with their
# category sequences (lowest → highest).  The order matters: OrdinalEncoder
# maps position 0 → 0, position 1 → 1, etc., preserving the natural ordering
# so that the encoded integers carry meaningful distance information.
# ---------------------------------------------------------------------------
ORDINAL_COLS_SPEC = [
    # Child education level
    ('CB5A', ['NOT ATTENDING', 'ECE', 'PRIMARY',
              'LOWER SECONDARY', 'UPPER SECONDARY', 'HIGHER']),
    # Mother's highest education level
    ('WB6A', ['NONE', 'ECE', 'PRIMARY', 'VOCATIONAL TRAINING',
              'LOWER SECONDARY', 'UPPER SECONDARY', 'HIGHER']),
    # Mother's literacy level
    ('WB14', ['CANNOT READ AT ALL',
              'NO SENTENCE IN REQUIRED LANGUAGE / BRAILLE',
              'ABLE TO READ ONLY PARTS OF SENTENCE',
              'ABLE TO READ WHOLE SENTENCE']),
    # Mother's adult-functioning difficulty (remembering, self-care, communicating)
    ('AF10', ['NO DIFFICULTY', 'SOME DIFFICULTY', 'A LOT OF DIFFICULTY']),
    ('AF11', ['NO DIFFICULTY', 'SOME DIFFICULTY', 'A LOT OF DIFFICULTY']),
    ('AF12', ['NO DIFFICULTY', 'SOME DIFFICULTY', 'A LOT OF DIFFICULTY']),
    # Mother's subjective well-being
    ('LS1', ['VERY UNHAPPY', 'SOMEWHAT UNHAPPY',
             'NEITHER HAPPY NOR UNHAPPY', 'SOMEWHAT HAPPY', 'VERY HAPPY']),
    ('LS3', ['WORSENED', 'MORE OR LESS THE SAME', 'IMPROVED']),
    ('LS4', ['WORSE',   'MORE OR LESS THE SAME', 'BETTER']),
    # Perceived neighbourhood safety (day / night)
    ('VT20', ['VERY UNSAFE', 'UNSAFE', 'NEVER WALK ALONE AFTER DARK', 'SAFE', 'VERY SAFE']),
    ('VT21', ['VERY UNSAFE', 'UNSAFE', 'NEVER WALK ALONE AFTER DARK', 'SAFE', 'VERY SAFE']),
    # Water sufficiency
    ('WS7', ['NO, ALWAYS SUFFICIENT', 'DK', 'YES, AT LEAST ONCE']),
    # Housing quality (floor / roof material – higher index ≈ higher quality)
    ('HC4', ['EARTH / SAND', 'DUNG', 'PALM / BAMBOO', 'OTHER',
             'WOOD PLANKS', 'VINYL OR ASPHALT STRIPS', 'CEMENT',
             'CARPET', 'PARQUET OR POLISHED WOOD', 'CERAMIC TILES']),
    ('HC5', ['NO ROOF', 'RUSTIC MAT', 'THATCH / PALM LEAF', 'PALM / BAMBOO',
             'WOOD', 'WOOD PLANKS', 'CALAMINE / CEMENT FIBRE',
             'ROOFING SHINGLES', 'OTHER', 'IRON SHEETS / METAL / TIN',
             'CEMENT', 'CERAMIC TILES']),
]
# Derived convenience lists (kept in sync automatically)
ORDINAL_COLS = [col   for col, _    in ORDINAL_COLS_SPEC]
ORDINAL_CATS = [cats  for _,   cats in ORDINAL_COLS_SPEC]

# ---------------------------------------------------------------------------
# BINARY_COLS – YES / NO variables (encoded to 0 / 1 in encode_binary).
# FCD2A-K : specific discipline methods used on the child
# FCD5    : physical punishment indicator
# DV1A-E  : mother's attitudes toward domestic violence (each item)
# VT22A-X : specific victimisation events experienced by mother
# HC*     : asset ownership (electricity, radio, TV, fridge, phone, car, computer)
# ---------------------------------------------------------------------------
BINARY_COLS = [
    # Child schooling & labour
    'CB4', 'CB7', 'CB11', 'CL2', 'CL12',
    # Child discipline methods (FCD2A–K) + severity flag
    'FCD2A', 'FCD2B', 'FCD2C', 'FCD2D', 'FCD2E', 'FCD2F',
    'FCD2G', 'FCD2H', 'FCD2I', 'FCD2J', 'FCD2K', 'FCD5',
    # Maternal background & marital status
    'WB5', 'MA3', 'disability',
    # Substance use
    'TA1', 'TA14',
    # Household assets
    'HC8', 'HC11', 'HC12', 'HC13', 'HC15', 'HC17', 'HC19', 'TN1',
    # Sanitation & victimisation
    'WS15', 'VT1', 'VT9',
    'VT22A', 'VT22B', 'VT22C', 'VT22D', 'VT22E', 'VT22F', 'VT22X',
    # Domestic violence attitudes
    'DV1A', 'DV1B', 'DV1C', 'DV1D', 'DV1E',
]

# ---------------------------------------------------------------------------
# ENGINEERED_COLS – composite features created in engineer_features().
# These are treated as continuous and scaled with StandardScaler.
# ---------------------------------------------------------------------------
ENGINEERED_COLS = [
    'mother_not_matched',     # 1 if mother data could not be linked (WB4 missing)
    'dv_acceptance_score',    # count of DV attitudes endorsed (0-5)
    'any_victimisation',      # 1 if any of VT22A-F is YES
    'water_improved',         # 1 if water source meets JMP 'improved' definition
    'sanitation_improved',    # 1 if toilet meets JMP 'improved' definition
    'asset_count',            # number of household assets owned (0-7)
    'discipline_count',       # number of discipline methods used (0-11)
    'total_labour_hours',     # CL3 + CL13 (paid + unpaid work hours)
    'mother_difficulty_score',# sum of AF10-AF12 mapped to 0/1/2 (0-6)
]

# ---------------------------------------------------------------------------
# NOMINAL_COLS – unordered categorical variables; one-hot encoded.
# HH6 / HH7 : urban/rural & district (geography)
# HL4        : child sex
# ethnicity  : ethnic group
# MSTATUS    : marital status of mother
# WS3 / WS14 : water / toilet facility location
# HW5        : handwashing station material
# HC14       : dwelling ownership status
# WB6B       : mother's field of education
# ---------------------------------------------------------------------------
NOMINAL_COLS = [
    'HH6', 'HH7', 'HL4', 'ethnicity', 'MSTATUS',
    'WS3', 'WS14', 'HW5', 'HC14', 'WB6B',
]

## Section 1 – Target Engineering

`FCF26` asks *"How often does the child seem very sad or depressed?"*
with responses: NEVER | A FEW TIMES A YEAR | MONTHLY | WEEKLY | DAILY.

We binarise to **0 = no depression (NEVER)** vs **1 = any depression** (all other responses),
per the project specification.


In [3]:
def build_target(df: pd.DataFrame) -> pd.Series:
    """
    Binarise the depression target variable FCF26.

    Parameters
    ----------
    df : Raw dataframe containing the 'FCF26' column.

    Returns
    -------
    pd.Series of dtype int (0 / 1), with NaN where FCF26 was missing / NO RESPONSE.
    """
    depression_map = {
        'NEVER':             0,
        'A FEW TIMES A YEAR': 1,
        'MONTHLY':           1,
        'WEEKLY':            1,
        'DAILY':             1,
        # 'NO RESPONSE' and any unrecognised value will map to NaN via .map()
    }
    return df['FCF26'].map(depression_map).rename('depression')


def drop_unusable_rows(df: pd.DataFrame, y: pd.Series):
    """
    Remove rows that cannot be used for modelling:
      - NaN target (FCF26 missing / NO RESPONSE, ~103 rows)
      - Missing child age CB3 (~23 rows)

    Parameters
    ----------
    df : Dataframe after raw load.
    y  : Binarised target series.

    Returns
    -------
    (df_clean, y_clean) with aligned indices.
    """
    valid_mask = y.notna() & df['CB3'].notna()
    return df.loc[valid_mask].copy(), y.loc[valid_mask].copy()

## Section 2 – Skip-Pattern Fixes (Pre-cleaning)

The MICS questionnaire uses **skip logic**: respondents who answered NO to a gating question
are skipped over follow-up questions, leaving those fields blank.  These structural missings
are *not* random – they carry information (e.g. `CL3 = NaN` means the child did zero paid
hours because they don't work).  We resolve each skip pattern explicitly before any
statistical imputation, so the imputer only sees genuinely unknown values.

Each lettered sub-section (A–L) corresponds to a questionnaire domain.


In [4]:
def fix_skip_patterns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Resolve survey skip-logic missings and standardise NO RESPONSE codes.

    All operations are performed on a copy – the input dataframe is not mutated.

    Parameters
    ----------
    df : Dataframe after drop_unusable_rows.

    Returns
    -------
    df_cleaned with resolved structural missings.
    """
    df = df.copy()

    # ------------------------------------------------------------------
    # A. CHILD LABOUR (CL3, CL13)
    # CL3  = paid labour hours per week (skipped when CL2 == 'NO')
    # CL13 = household work hours per week (skipped when CL12 == 'NO')
    # Structural missing → 0 hours worked.
    # ------------------------------------------------------------------
    df['CL3']  = df['CL3'].fillna(0)
    df['CL13'] = pd.to_numeric(df['CL13'], errors='coerce').fillna(0)

    # ------------------------------------------------------------------
    # B. CHILD DISCIPLINE (FCD2A–K, FCD5)
    # These questions are asked only for children aged 5–14, and within
    # that group only when at least one discipline method was used.
    # Skipped → 'NO' (method was not used / not applicable).
    # 'NO RESPONSE' is also recoded to 'NO' as a conservative assumption.
    # ------------------------------------------------------------------
    fcd_method_cols = [f'FCD2{letter}' for letter in 'ABCDEFGHIJK']
    for col in fcd_method_cols:
        df[col] = df[col].fillna('NO').replace('NO RESPONSE', 'NO')

    # FCD5 = physical punishment indicator; DK / NO OPINION treated as NO
    df['FCD5'] = (df['FCD5']
                  .fillna('NO')
                  .replace({'DK / NO OPINION': 'NO', 'NO RESPONSE': 'NO'}))

    # ------------------------------------------------------------------
    # C. CHILD SCHOOLING (CB4, CB5A, CB5B, CB7, CB11)
    # CB5A / CB5B are skipped when CB4 == 'NO' (not attending school).
    # CB5B (grade) is mapped to an integer and dropped in favour of CB5B_num.
    # ------------------------------------------------------------------
    df['CB4']  = df['CB4'].replace('NO RESPONSE', 'NO')
    df['CB5A'] = df['CB5A'].fillna('NOT ATTENDING').replace('NO RESPONSE', 'NOT ATTENDING')

    grade_map = {
        'CLASS/YEAR/GRADE 1': 1, 'CLASS/YEAR/GRADE 2': 2,
        'CLASS/YEAR/GRADE 3': 3, 'CLASS/YEAR/GRADE 4': 4,
        'CLASS/YEAR/GRADE 5': 5, 'CLASS/GRADE 6':      6,
        'CLASS/GRADE 7':      7, 'CLASS/GRADE 8':      8,
    }
    # Children not attending school receive grade 0
    df['CB5B_num'] = df['CB5B'].map(grade_map).fillna(0).astype(int)
    df = df.drop(columns=['CB5B'])

    df['CB7']  = df['CB7'].fillna('NO')
    df['CB11'] = df['CB11'].fillna('NO').replace('NO RESPONSE', 'NO')

    # ------------------------------------------------------------------
    # D. MATERNAL EDUCATION (WB5, WB6A, WB6B, WB14)
    # WB14 (literacy) is skipped for women with secondary+ education;
    # they are assumed to be fully literate.
    # ------------------------------------------------------------------
    df['WB5']  = df['WB5'].replace('NO RESPONSE', 'NO')
    df['WB6A'] = df['WB6A'].fillna('NONE')

    high_ed_mask = df['WB6A'].isin(
        ['LOWER SECONDARY', 'UPPER SECONDARY', 'HIGHER', 'VOCATIONAL TRAINING']
    )
    df.loc[high_ed_mask & df['WB14'].isna(), 'WB14'] = 'ABLE TO READ WHOLE SENTENCE'
    df['WB14'] = df['WB14'].replace('NO RESPONSE', 'CANNOT READ AT ALL')

    # ------------------------------------------------------------------
    # E. HUSBAND / PARTNER FIELDS (MA2, MA3, MSTATUS)
    # MA2 / MA3 are skipped when the woman is not currently married.
    # Unmarried → husband age = 0, multiple wives = NO.
    # '9.0' in MSTATUS is a data artefact (coded missing); set to NaN.
    # ------------------------------------------------------------------
    df['MSTATUS'] = df['MSTATUS'].replace('9.0', np.nan)
    df['MA2']     = pd.to_numeric(df['MA2'], errors='coerce')

    not_married_mask = df['MSTATUS'] != 'Currently married/in union'
    df.loc[not_married_mask & df['MA2'].isna(), 'MA2'] = 0
    df.loc[not_married_mask & df['MA3'].isna(), 'MA3'] = 'NO'
    df['MA3'] = df['MA3'].replace('NO RESPONSE', 'NO')

    # ------------------------------------------------------------------
    # F. TOBACCO & ALCOHOL (TA1, TA14)
    # NO RESPONSE treated as NO (conservative / most common category).
    # ------------------------------------------------------------------
    df['TA1']  = df['TA1'].replace('NO RESPONSE', 'NO')
    df['TA14'] = df['TA14'].replace('NO RESPONSE', 'NO')

    # ------------------------------------------------------------------
    # G. LIFE SATISFACTION (LS1, LS2, LS3, LS4)
    # NO RESPONSE is genuinely unknown here → NaN (handled later by imputer).
    # LS2 is a numeric rating scale; coerce to float.
    # ------------------------------------------------------------------
    for col in ['LS1', 'LS3', 'LS4']:
        df[col] = df[col].replace('NO RESPONSE', np.nan)
    df['LS2'] = pd.to_numeric(df['LS2'].replace('NO RESPONSE', np.nan), errors='coerce')

    # ------------------------------------------------------------------
    # H. ADULT FUNCTIONING (AF10–AF12)
    # AF12 NO RESPONSE → NaN; remaining missing values imputed later.
    # ------------------------------------------------------------------
    df['AF12'] = df['AF12'].replace('NO RESPONSE', np.nan)

    # ------------------------------------------------------------------
    # I. WATER & SANITATION (WS3, WS4, WS14, WS15)
    # WS3 (water collection location) is skipped when the source is
    # on-premises (piped into dwelling or yard) → impute 'IN OWN DWELLING'.
    # WS4 (time to water source in minutes) is 0 when WS3 is on-premises.
    # WS14 / WS15 are skipped when there is no facility at all.
    # ------------------------------------------------------------------
    on_premises_water = df['WS1'].isin(
        ['PIPED WATER: PIPED INTO DWELLING', 'PIPED WATER: PIPED TO YARD / PLOT']
    )
    df.loc[on_premises_water & df['WS3'].isna(), 'WS3'] = 'IN OWN DWELLING'
    df['WS3'] = df['WS3'].fillna('ELSEWHERE').replace('NO RESPONSE', 'ELSEWHERE')

    df['WS4'] = pd.to_numeric(df['WS4'], errors='coerce')
    on_premises_loc = df['WS3'].isin(['IN OWN DWELLING', 'IN OWN YARD / PLOT'])
    df.loc[on_premises_loc & df['WS4'].isna(), 'WS4'] = 0

    no_facility = df['WS11'] == 'NO FACILITY / BUSH / FIELD'
    df.loc[no_facility & df['WS14'].isna(), 'WS14'] = 'NO FACILITY'
    df.loc[no_facility & df['WS15'].isna(), 'WS15'] = 'NO'
    df['WS14'] = df['WS14'].fillna('ELSEWHERE').replace('NO RESPONSE', 'ELSEWHERE')
    df['WS15'] = df['WS15'].fillna('NO').replace('NO RESPONSE', 'NO')

    # ------------------------------------------------------------------
    # J. HOUSEHOLD ASSETS (HC8, HC11–HC13, HC14, HC15, HC17, HC19, TN1)
    # NO RESPONSE → NO (asset not owned).
    # HC8 has two YES variants (grid / off-grid); merge into single YES.
    # HC14 = dwelling ownership; NO RESPONSE → OWN (most common category).
    # ------------------------------------------------------------------
    asset_binary_cols = ['HC8', 'HC11', 'HC12', 'HC13', 'HC15', 'HC17', 'HC19', 'TN1']
    for col in asset_binary_cols:
        df[col] = df[col].replace('NO RESPONSE', 'NO')

    df['HC8'] = df['HC8'].replace({
        'YES, INTERCONNECTED GRID':                    'YES',
        'YES, OFF-GRID (GENERATOR/ISOLATED SYSTEM)':   'YES',
    })
    df['HC14'] = df['HC14'].replace('NO RESPONSE', 'OWN')

    # ------------------------------------------------------------------
    # K. HANDWASHING (HW5)
    # Missing → 'NO STATION' (no dedicated handwashing facility).
    # NO RESPONSE → 'NO' (no soap / detergent available).
    # ------------------------------------------------------------------
    df['HW5'] = df['HW5'].fillna('NO STATION').replace('NO RESPONSE', 'NO')

    # ------------------------------------------------------------------
    # L. DOMESTIC VIOLENCE & VICTIMISATION (DV1A–E, VT1, VT9, VT20–21, VT22A–X)
    # These are sensitive questions with genuine non-response; set to NaN
    # so that the geographic group imputer (Section 4) handles them.
    # ------------------------------------------------------------------
    dv_vt_cols = (
        [f'DV1{x}' for x in 'ABCDE']
        + ['VT1', 'VT9', 'VT20', 'VT21']
        + [f'VT22{x}' for x in list('ABCDEF') + ['X']]
    )
    for col in dv_vt_cols:
        df[col] = df[col].replace('NO RESPONSE', np.nan)

    return df

## Section 3 – Composite Feature Engineering

Domain-driven aggregate features that are expected to have stronger predictive signal
than the individual raw items.  All features are derived from variables already present
in the dataset; no external data sources are required.


In [5]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create composite features from existing columns.

    All new columns are listed in ENGINEERED_COLS and processed by the
    'engineered' transformer in the sklearn ColumnTransformer.

    Parameters
    ----------
    df : Dataframe after fix_skip_patterns.

    Returns
    -------
    df with additional engineered columns appended.
    """
    df = df.copy()

    # mother_not_matched
    # WB4 (mother's age) is NaN when mother data could not be linked to the child
    # record.  This flag captures that missingness as an explicit feature.
    df['mother_not_matched'] = df['WB4'].isna().astype(int)

    # dv_acceptance_score
    # Count of DV attitude items (DV1A–E) where mother answered YES
    # (i.e. endorses domestic violence as acceptable). Range: 0–5.
    dv_cols = [f'DV1{x}' for x in 'ABCDE']
    df['dv_acceptance_score'] = df[dv_cols].eq('YES').sum(axis=1)

    # any_victimisation
    # 1 if the mother reported any of six victimisation events (VT22A–F).
    vt_specific_cols = [f'VT22{x}' for x in 'ABCDEF']
    df['any_victimisation'] = df[vt_specific_cols].eq('YES').any(axis=1).astype(int)

    # water_improved
    # 1 if the household's drinking water source meets the WHO/UNICEF JMP
    # definition of an 'improved' source.
    improved_water_sources = {
        'PIPED WATER: PIPED INTO DWELLING',
        'PIPED WATER: PIPED TO YARD / PLOT',
        'PIPED WATER: PIPED TO NEIGHBOUR',
        'PIPED WATER: PUBLIC TAP / STANDPIPE',
        'TUBE WELL / BOREHOLE',
        'DUG WELL: PROTECTED WELL',
        'SPRING: PROTECTED SPRING',
        'RAINWATER',
        'PACKAGED WATER: BOTTLED WATER',
        'WATER KIOSK',
    }
    df['water_improved'] = df['WS1'].isin(improved_water_sources).astype(int)

    # sanitation_improved
    # 1 if the household uses an 'improved' sanitation facility (JMP definition).
    improved_sanitation_facilities = {
        'FLUSH / POUR FLUSH: FLUSH TO PIPED SEWER SYSTEM',
        'FLUSH / POUR FLUSH: FLUSH TO SEPTIC TANK',
        'FLUSH / POUR FLUSH: FLUSH TO PIT LATRINE',
        'PIT LATRINE: VENTILATED IMPROVED PIT LATRINE',
        'PIT LATRINE: PIT LATRINE WITH SLAB',
        'COMPOSTING TOILET',
    }
    df['sanitation_improved'] = df['WS11'].isin(improved_sanitation_facilities).astype(int)

    # asset_count
    # Number of durable household assets owned (electricity, radio, TV,
    # fridge, mobile phone, non-mobile phone, bicycle, car, computer).
    # Range: 0–7. Proxy for household wealth beyond the continuous wscore.
    asset_cols = ['HC8', 'HC11', 'HC12', 'HC13', 'HC15', 'HC17', 'HC19']
    df['asset_count'] = df[asset_cols].eq('YES').sum(axis=1)

    # discipline_count
    # Number of distinct discipline methods (FCD2A–K) applied to the child.
    # Range: 0–11. Higher counts indicate more frequent / varied discipline.
    fcd_method_cols = [f'FCD2{x}' for x in 'ABCDEFGHIJK']
    df['discipline_count'] = df[fcd_method_cols].eq('YES').sum(axis=1)

    # total_labour_hours
    # Combined weekly hours of paid (CL3) and unpaid household (CL13) work.
    # Already set to 0 for non-working children in fix_skip_patterns.
    df['total_labour_hours'] = df['CL3'] + df['CL13']

    # mother_difficulty_score
    # Sum of mother's self-reported difficulty levels across three domains:
    # AF10 (remembering), AF11 (self-care), AF12 (communicating).
    # Mapped: NO DIFFICULTY=0, SOME DIFFICULTY=1, A LOT OF DIFFICULTY=2.
    # Range: 0–6. Proxy for maternal cognitive / functional impairment.
    difficulty_level_map = {'NO DIFFICULTY': 0, 'SOME DIFFICULTY': 1, 'A LOT OF DIFFICULTY': 2}
    df['mother_difficulty_score'] = (
        df[['AF10', 'AF11', 'AF12']]
        .apply(lambda col: col.map(difficulty_level_map))
        .sum(axis=1)
    )

    return df


# Wrap as an sklearn FunctionTransformer so it can be used inside a Pipeline
FeatureEngineer = FunctionTransformer(engineer_features, validate=False)

## Section 4 – Group Imputation

Remaining missing values after Section 2 are genuinely unknown.  A simple geographic
group imputation (district × urban/rural) is preferred over global imputation because
Malawi shows strong regional variation in socioeconomic indicators.

**Leakage prevention**: fill values are computed *only* from the training split and then
applied to both train and test.


In [6]:
def group_impute(
    train: pd.DataFrame,
    test:  pd.DataFrame,
    col:   str,
    group_cols: list,
    strategy: str = 'mode',
):
    """
    Impute missing values in `col` using the group statistic computed on `train`.

    Parameters
    ----------
    train, test  : DataFrames sharing the same column structure.
    col          : Column to impute.
    group_cols   : Columns to group by when computing fill values.
    strategy     : 'mode' for categorical columns, 'median' for numeric.

    Returns
    -------
    (train, test) with `col` imputed in-place.

    Notes
    -----
    - Fill values are derived exclusively from `train` to prevent data leakage.
    - If a group has no observed values (all NaN), the global statistic is used
      as a fallback.
    """
    # Compute group fill values from training data only
    if strategy == 'mode':
        group_fills = train.groupby(group_cols)[col].agg(
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        )
        global_fill = train[col].mode().iloc[0] if not train[col].mode().empty else np.nan
    else:  # median
        group_fills = train.groupby(group_cols)[col].median()
        global_fill = train[col].median()

    def _fill_series(df_part: pd.DataFrame) -> pd.DataFrame:
        """Apply group fill values to a single dataframe partition."""
        missing_mask = df_part[col].isna()
        if not missing_mask.any():
            return df_part
        for idx in df_part[missing_mask].index:
            group_key = tuple(df_part.loc[idx, g] for g in group_cols)
            df_part.loc[idx, col] = group_fills.get(group_key, global_fill)
        return df_part

    train = _fill_series(train)
    test  = _fill_series(test)
    return train, test


def apply_all_imputations(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """
    Apply the full set of group-imputation rules to train and test splits.

    Each rule is a tuple: (column, group_cols, strategy).
    Rules are applied in order; earlier imputations can be used as
    grouping keys for later ones (e.g. WB6A is imputed before WB14).

    Parameters
    ----------
    X_train, X_test : DataFrames from train_test_split.

    Returns
    -------
    (X_train, X_test) with all specified columns imputed.
    """
    GEO = ['HH7', 'HH6']  # district + urban/rural – primary stratification variables

    imputation_rules = [
        # --- Maternal background ---
        ('WB4',  GEO,          'median'),  # mother's age
        ('WB5',  GEO,          'mode'),    # mother attended school
        ('WB6A', GEO,          'mode'),    # mother's highest education level
        ('WB6B', GEO,          'mode'),    # mother's field of education
        ('WB14', ['WB6A'],     'mode'),    # mother's literacy (grouped by education)

        # --- Domestic violence attitudes (grouped by geography) ---
        *[(f'DV1{x}', GEO, 'mode') for x in 'ABCDE'],

        # --- Victimisation ---
        ('VT1',  GEO, 'mode'),
        ('VT9',  GEO, 'mode'),
        ('VT20', GEO, 'mode'),
        ('VT21', GEO, 'mode'),
        *[(f'VT22{x}', GEO, 'mode') for x in list('ABCDEF') + ['X']],

        # --- Marriage / union ---
        ('MSTATUS', GEO,               'mode'),    # marital status
        ('MA2',     ['MSTATUS', 'HH7'], 'median'),  # husband's age
        ('MA3',     ['MSTATUS', 'HH7'], 'mode'),    # multiple wives

        # --- Adult functioning ---
        ('disability', GEO, 'mode'),
        ('AF10',       GEO, 'mode'),
        ('AF11',       GEO, 'mode'),
        ('AF12',       GEO, 'mode'),

        # --- Substance use ---
        ('TA1',  GEO, 'mode'),
        ('TA14', GEO, 'mode'),

        # --- Life satisfaction ---
        ('LS1', GEO, 'mode'),
        ('LS2', GEO, 'median'),
        ('LS3', GEO, 'mode'),
        ('LS4', GEO, 'mode'),

        # --- Water & sanitation ---
        ('WS4', GEO, 'median'),  # minutes to water source
        ('WS7', GEO, 'mode'),    # water sufficiency
    ]

    for col, group_cols, strategy in imputation_rules:
        if col in X_train.columns:
            X_train, X_test = group_impute(X_train, X_test, col, group_cols, strategy)

    return X_train, X_test

## Section 5 – Binary Encoding

Converts YES/NO string columns to integers (0/1) before the sklearn pipeline.
This is done outside sklearn because the conversion rules are heterogeneous
(standard YES/NO vs. the `disability` field which uses a descriptive string).


In [7]:
def encode_binary(df: pd.DataFrame, binary_cols: list) -> pd.DataFrame:
    """
    Convert YES / NO string columns to integer (0 / 1).

    Handles three encoding cases:
      1. Boolean dtype  – cast directly.
      2. 'disability'   – special string ('Has functional difficulty' → 1).
      3. All others     – string == 'YES' (case-insensitive) → 1.

    Parameters
    ----------
    df          : Dataframe with binary string columns.
    binary_cols : List of column names to encode.

    Returns
    -------
    df with specified columns converted to int dtype.
    """
    df = df.copy()
    for col in binary_cols:
        if col not in df.columns:
            continue
        if df[col].dtype == bool:
            df[col] = df[col].astype(int)
        elif col == 'disability':
            df[col] = (df[col] == 'Has functional difficulty').astype(int)
        else:
            df[col] = (df[col].astype(str).str.upper() == 'YES').astype(int)
    return df

## Section 6 – Sklearn Preprocessing Pipeline

A `ColumnTransformer` applies the appropriate transformation to each feature group:

| Transformer | Columns | Steps |
|---|---|---|
| `numeric_pipe`    | NUMERIC_COLS    | Median impute → StandardScaler |
| `ordinal_pipe`    | ORDINAL_COLS    | Mode impute → OrdinalEncoder → StandardScaler |
| `binary_pipe`     | BINARY_COLS     | Mode impute (already 0/1 integers) |
| `engineered_pipe` | ENGINEERED_COLS | Median impute → StandardScaler |
| `nominal_pipe`    | NOMINAL_COLS    | Mode impute → OneHotEncoder |

`remainder='drop'` ensures that any columns not explicitly listed are excluded,
preventing accidental inclusion of ID columns (HH1, HH2, LN, FS4).


In [8]:
def preprocessing_pipeline() -> ColumnTransformer:
    """
    Build and return the sklearn ColumnTransformer for preprocessing.

    Returns
    -------
    ColumnTransformer (unfitted).  Fit it inside a full Pipeline to
    ensure no leakage between train and test splits.
    """
    # Continuous numeric features: impute then standardise
    numeric_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ])

    # Ordinal categorical: impute → encode preserving order → standardise
    # StandardScaler on ordinal integers keeps the scale comparable to numerics.
    ordinal_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(
            categories=ORDINAL_CATS,
            handle_unknown='use_encoded_value',
            unknown_value=-1,   # unseen categories during test → -1
        )),
        ('scale',  StandardScaler()),
    ])

    # Binary features: already 0/1 integers from encode_binary;
    # only a fallback imputer is needed for any residual NaN
    binary_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
    ])

    # Engineered aggregate features: treated as numeric
    engineered_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ])

    # Nominal (unordered) categorical: impute then one-hot encode
    # handle_unknown='ignore' silently zeros out any unseen category at test time
    nominal_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(
            drop=None,
            sparse_output=False,
            handle_unknown='ignore',
        )),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('numeric',    numeric_pipe,    NUMERIC_COLS),
            ('ordinal',    ordinal_pipe,    ORDINAL_COLS),
            ('binary',     binary_pipe,     BINARY_COLS),
            ('engineered', engineered_pipe, ENGINEERED_COLS),
            ('nominal',    nominal_pipe,    NOMINAL_COLS),
        ],
        remainder='drop',                   # exclude ID and unused columns
        verbose_feature_names_out=False,    # keep original feature names in output
    )
    return preprocessor

## Section 7 – `build_Xy` Orchestrator

Single entry point that runs the full pipeline in order and returns
train/test splits ready for an sklearn model pipeline.


In [9]:
def build_Xy(
    csv_path:     str,
    test_size:    float = 0.2,
    random_state: int   = 42,
    verbose:      bool  = True,
):
    """
    Full preprocessing pipeline: raw CSV → train/test feature matrices.

    Steps executed in order
    -----------------------
    1. Load raw CSV.
    2. Build binary target (build_target).
    3. Drop rows with missing target or child age (drop_unusable_rows).
    4. Resolve survey skip patterns (fix_skip_patterns).
    5. Create composite features (engineer_features).
    6. Stratified train / test split.
    7. Geographic group imputation – fit on train, apply to both (apply_all_imputations).
    8. Binary encoding: YES/NO → 0/1 (encode_binary).

    Parameters
    ----------
    csv_path     : Path to unicef_malawi.csv.
    test_size    : Fraction of data held out for evaluation (default 0.2).
    random_state : Random seed for reproducibility (default 42).
    verbose      : Print progress and shape information when True.

    Returns
    -------
    X_train, X_test, y_train, y_test
        DataFrames / Series ready to be passed to an sklearn Pipeline.
    """
    # Step 1 – Load
    df = pd.read_csv(csv_path)
    if verbose:
        print(f'[1] Raw data loaded:        {df.shape}')

    # Step 2 – Target
    y = build_target(df)

    # Step 3 – Drop unusable rows
    df, y = drop_unusable_rows(df, y)
    if verbose:
        print(f'[2] After row drops:        {df.shape}'
              f'  (class 0: {(y == 0).sum()}, class 1: {(y == 1).sum()})')

    # Step 4 – Skip-pattern fixes
    df = fix_skip_patterns(df)
    if verbose:
        print(f'[3] Skip patterns resolved: {df.shape}')

    # Step 5 – Feature engineering
    df = engineer_features(df)
    if verbose:
        print(f'[4] Features engineered:    {df.shape}')

    # Step 6 – Stratified train / test split
    # Stratify on y to preserve class balance in both splits.
    X_train, X_test, y_train, y_test = train_test_split(
        df, y,
        test_size=test_size,
        shuffle=True,
        random_state=random_state,
        stratify=y,
    )
    if verbose:
        print(f'[5] Train / test split:     train={X_train.shape}, test={X_test.shape}')

    # Step 7 – Group imputation (train-fitted, applied to both)
    X_train, X_test = apply_all_imputations(X_train, X_test)
    if verbose:
        remaining_na = X_train.isna().sum().sum()
        print(f'[6] Group imputation done.  Remaining NaN in X_train: {remaining_na}')

    # Step 8 – Binary encoding
    # Encode BINARY_COLS plus the FCD2 discipline method columns
    all_binary_cols = BINARY_COLS + [f'FCD2{x}' for x in 'ABCDEFGHIJK']
    X_train = encode_binary(X_train, all_binary_cols)
    X_test  = encode_binary(X_test,  all_binary_cols)
    if verbose:
        print('[7] Binary encoding done.')

    return X_train, X_test, y_train, y_test

## Section 8 – Model Training Demo

Quick sanity-check demonstrating that the pipeline produces valid inputs for two baseline
classifiers.  Full model selection, hyperparameter tuning, and evaluation belong in
`project.ipynb`.


In [10]:
# ── Run pipeline ─────────────────────────────────────────────────────────────
CSV_PATH = 'unicef_malawi.csv'

X_train, X_test, y_train, y_test = build_Xy(CSV_PATH, verbose=True)

# ── Build model pipelines ─────────────────────────────────────────────────────
# Each pipeline chains the shared preprocessor with a classifier so that
# the ColumnTransformer is always fitted on training data only.

lr_pipe = Pipeline([
    ('preprocess', preprocessing_pipeline()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

rf_pipe = Pipeline([
    ('preprocess', preprocessing_pipeline()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)),
])

# ── Fit ───────────────────────────────────────────────────────────────────────
lr_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
n_features = len(lr_pipe['preprocess'].get_feature_names_out())

print(f'\n{"="*60}')
print(f'Total features after preprocessing: {n_features}')
print(f'{"="*60}')

for name, pipe in [('Logistic Regression', lr_pipe), ('Random Forest', rf_pipe)]:
    train_acc = pipe.score(X_train, y_train)
    test_acc  = pipe.score(X_test,  y_test)
    print(f'\n{name}')
    print(f'  Train accuracy : {train_acc:.4f}')
    print(f'  Test  accuracy : {test_acc:.4f}')
    print(classification_report(
        y_test, pipe.predict(X_test),
        target_names=['No depression', 'Depression'],
    ))

[1] Raw data loaded:        (13162, 87)
[2] After row drops:        (13036, 87)  (class 0: 5905, class 1: 7131)
[3] Skip patterns resolved: (13036, 87)
[4] Features engineered:    (13036, 96)
[5] Train / test split:     train=(10428, 96), test=(2608, 96)
[6] Group imputation done.  Remaining NaN in X_train: 0
[7] Binary encoding done.

Total features after preprocessing: 120

Logistic Regression
  Train accuracy : 0.6169
  Test  accuracy : 0.5982
               precision    recall  f1-score   support

No depression       0.57      0.46      0.51      1181
   Depression       0.61      0.72      0.66      1427

     accuracy                           0.60      2608
    macro avg       0.59      0.59      0.58      2608
 weighted avg       0.59      0.60      0.59      2608


Random Forest
  Train accuracy : 1.0000
  Test  accuracy : 0.5955
               precision    recall  f1-score   support

No depression       0.57      0.45      0.50      1181
   Depression       0.61      0.72    